# Analysis of the UNESCO "Cities of Music" against each country's music exports and imports (2006-2023)

## Advanced Visualization

## What are the UNESCO "Cities of Music"?
The UNESCO "Cities of Music" programme was born in 2006 as part of the UNESCO Creative Cities Network. Its aim is to recognize cities around the world that stand out for acknowledging and promoting the value of music as a factor of cultural and social understanding, and also for its importance to a city's economic development and identity.

In the broader context, the UNESCO Creative Cities Network seeks to promote cooperation with and among cities that identify creativity as a fundamental strategic component of sustainable urban development.

## Approach and analysis proposal
With that in mind, I set out to analyze the changes, variations and trends in the global market for creative goods (focused on music) over time, taking the designations of these "Cities of Music" as "milestones".

### To answer the following questions:
- What place does the creative goods market occupy within the broader context of global goods imports/exports? And within the creative goods market, what place have music goods occupied over the years studied?

- Do a country's music import/export shares change after these designations? Or are the designations rather the result of upward or downward trends in the creative goods markets?

- Through cartographic visualization, can any spatial pattern be observed in the designation of these "Cities of Music"? Is there a greater concentration of cities in some region or continent?

- Is the export or the import of music goods more relevant in the formation of these music "hubs"?

## Data sources (2006-2023):
The period of analysis runs from 2006, when the first "City of Music" designations were made, to 2023, the last year for which UNCTAD (UN Trade & Development) has data on international trade in creative goods (18 years). The "Cities of Music" programme continues to make designations through 2025 and everything indicates that it will keep doing so.
Regarding the UNCTAD data, although the website states that the databases I consulted were updated on 10 February 2025, there is still no creative goods data for 2024 and 2025. In addition, for some countries, such as Cabo Verde, export/import information is not available for all years.

### Global goods imports/exports:
https://unctadstat.unctad.org/datacentre/

### Creative industry imports/exports (including music), global and by country:
https://unctad.org/creative-economy-programme/statistics

This refers to: concerts, albums, song rights, music platforms, etc. In the UNCTAD classification used here, the "music" series is the product group "Music, performing and visual arts" (its values are identical to that group in the products file).

### Cities awarded the title "City of Music" by UNESCO (2006-2023):
https://citiesofmusic.net/music-cities/

# Procedure and difficulties
For *data import, handling and processing* I used the `pandas` library.

For mathematical calculations (for example, logarithms): `numpy`.

For the *statistical charts* I used several `plotly` implementations (express, subplots, colors...).

I built the *choropleth maps* with `plotly.graph_objects` **for the layered map**, `geopy` (conversion of city/country names to latitude/longitude coordinates) and, for further refinement, a public GeoJSON file of country borders (loaded with `json` and `urllib`).

1. **Data cleaning and consolidation**:
    Although the information source is in every case the UN, its different bodies (even within UNESCO itself) use different country names, which is a first difficulty.
2. It was necessary to build **reference-unification dictionaries** so that no information was lost when charting or mapping, since some countries had different names.
3. Some countries, such as Palestine, had been left out of the maps even though one of their cities has been named "City of Music", so it was necessary to **refine the mapping methods**, referring to countries by codes and coordinates, not only by name.

# 0. Installs and imports required to run the notebook correctly:

In [ ]:
#!pip install pandas numpy plotly geopy nbformat

In [ ]:
# General imports. Some charts re-import what they need so that they work on their own.
# When that is the case, the import is included at the top of the snippet.

# --- DATA MANIPULATION LIBRARIES ---
import pandas as pd
import numpy as np
import json
from urllib.request import urlopen

# --- VISUALIZATION LIBRARIES ---
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# --- GEOSPATIAL LIBRARIES ---
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# 1. Data import, cleaning and corrections of country names

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors

# 1. LOAD FILES
try:
    goods_exp = pd.read_csv('data/goods_exps_USGoodsAndServicesBpm6_20260216_143950.csv')
    goods_imp = pd.read_csv('data/goods_imps_USGoodsAndServicesBpm6_20260216_144004.csv')
    tot_exp = pd.read_csv('data/tot_exp_USCreativeGoodsValue_20260216_130603.csv')
    tot_imp = pd.read_csv('data/tot_imp_USCreativeGoodsValue_20260216_124750.csv')
    mus_exp = pd.read_csv('data/mus_exp_USCreativeGoodsValue_20260216_124931.csv')
    mus_imp = pd.read_csv('data/mus_imp_USCreativeGoodsValue_20260216_125027.csv')
    cities_music = pd.read_csv('data/cities_of_music_2006-2023.csv')
    products = pd.read_csv('data/products_USCreativeGoodsValue_20260217_101031.csv')
    print("Files loaded successfully.")
except FileNotFoundError as e:
    print(f"File error: {e}")

# 2. DICTIONARY OF CORRECTIONS to make the Music and Trade DataFrames match
country_fixes = {
    'Türkiye': 'Turkiye',
    'Turkey': 'Turkiye',
    'United States of America': 'United States',
    'USA': 'United States',
    'Democratic Republic of Congo': 'Dem. Rep. of the Congo',
    'Democratic Republic of the Congo': 'Dem. Rep. of the Congo',
    'Bosnia & Herzegovina': 'Bosnia and Herzegovina',
    'South Korea': 'Republic of Korea',
    'Korea, Rep.': 'Republic of Korea',
    'Russia': 'Russian Federation',
    'Czech Republic': 'Czechia',
    'Iran': 'Iran (Islamic Republic of)',
    'Palestine': 'State of Palestine',
    'Palestinian Territories': 'State of Palestine',
    'Vietnam': 'Viet Nam',
    'Cape Verde': 'Cabo Verde',
    'Great Britain': 'United Kingdom',
    'Britain': 'United Kingdom',
    'Venezuela': 'Venezuela (Bolivarian Rep. of)',
    'Bolivia': 'Bolivia (Plurinational State of)',
    'Guinea Bissau': 'Guinea-Bissau',
    'Guinea-Bissau': 'Guinea-Bissau'
}

# 3. CLEANING
def clean_df(df, country_col='Economy_Label', year_col='Year'):
    df = df.copy()
    if country_col in df.columns:
        # Basic cleaning
        df[country_col] = df[country_col].astype(str).str.replace(r'\xa0', ' ', regex=True).str.strip()
        
        # Normalization and correction
        df['Country_Normalized'] = df[country_col].replace(country_fixes)
        
    if year_col in df.columns:
        df[year_col] = pd.to_numeric(df[year_col], errors='coerce')
    return df

mus_exp = clean_df(mus_exp)
mus_imp = clean_df(mus_imp)
goods_exp = clean_df(goods_exp)
tot_exp = clean_df(tot_exp)
cities_music = clean_df(cities_music, country_col='Country', year_col='Year')
products = clean_df(products)

print("Cleaning completed. Names synchronized successfully.")

# 4. FINAL CHECK
music_countries = set(cities_music['Country_Normalized'].unique())
trade_countries = set(mus_exp['Country_Normalized'].unique())
matched_countries = music_countries.intersection(trade_countries)
print(f"Matching countries: {len(matched_countries)}")

# 2. Initial exploration of trade shares in exports:
###      Trade of:
###      1) Global creative goods over total global goods
###      2) Music goods over global creative goods

From this first graphical exploration we find that global trade in creative goods is around 3% a year of total goods trade, while exports of music goods account for around 6% a year of global creative goods trade (between 4.6% and 7.0% depending on the year). That is, within the creative industries, music accounts for almost double the share that the creative industries as a whole hold in total global trade.

Although this sector of international trade is not impressive in economic size, it is very relevant for aspects such as those that give meaning to the "Cities of Music".

In [ ]:
# --- A. DATA PREPARATION (SHARES) ---

# 1. Creative goods vs total goods (exports)
exp_tot_goods = goods_exp.merge(tot_exp, on=['Economy_Label', 'Year'])
exp_tot_goods.rename(columns={
    'US_at_current_prices_in_millions_Value_x': 'goods_exp',
    'US_at_current_prices_in_millions_Value_y': 'creative_exp'
}, inplace=True)

exp_tot_goods['Prop_Creative_in_Goods'] = np.where(
    exp_tot_goods['goods_exp'] != 0,
    exp_tot_goods['creative_exp'] / exp_tot_goods['goods_exp'], np.nan
)

# 2. Music vs creative goods (exports)
exp_tot_mus = tot_exp.merge(mus_exp, on=['Economy_Label', 'Year'])
exp_tot_mus.rename(columns={
    'US_at_current_prices_in_millions_Value_x': 'creative_exp',
    'US_at_current_prices_in_millions_Value_y': 'music_exp'
}, inplace=True)

exp_tot_mus['Prop_Music_in_Creative'] = np.where(
    exp_tot_mus['creative_exp'] != 0,
    exp_tot_mus['music_exp'] / exp_tot_mus['creative_exp'], np.nan
)

# Keep only WORLD data
world_goods = exp_tot_goods[exp_tot_goods['Economy_Label'] == 'World']
world_mus = exp_tot_mus[exp_tot_mus['Economy_Label'] == 'World']

# --- B. COMPARATIVE CHART (SUBPLOTS) ---
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Exports of: Creative Goods / Total Goods", "Exports of: Music Goods / Creative Goods"),
    shared_yaxes=False
)

# Line 1
fig.add_trace(go.Scatter(
    x=world_goods['Year'], y=world_goods['Prop_Creative_in_Goods'],
    mode='lines+markers', name='Creative/Total',
    hovertemplate='%{y:.1%}'
), row=1, col=1)

# Line 2
fig.add_trace(go.Scatter(
    x=world_mus['Year'], y=world_mus['Prop_Music_in_Creative'],
    mode='lines+markers', name='Music/Creative',
    hovertemplate='%{y:.1%}'
), row=1, col=2)

fig.update_layout(title_text="Global Context: Shares (World)", template='simple_white', showlegend=False)
fig.update_yaxes(tickformat=".1%")
fig.show()

# 2a. Annual breakdown of exports by type of creative product (2002-2023)
This animated chart shows how traditional segments of creative goods, such as global exports of visual/audiovisual arts, printed publications and music, have grown very modestly or even declined in recent years (between 2008 and 2023, for example, books and publishing fell by about a third and music, performing and visual arts grew by only 18%). Meanwhile, others such as jewellery (roughly 2.5 times larger over the same period) and the very broad "Manufacturing of crafts and design goods" have been growing considerably for many years.

Two points emerge from this analysis:
1. It explains the need for initiatives such as "Cities of Music", whose interest is focused not only on the economic side of the music industry but on its social, cultural, urban and development impact.
2. It may be interesting to remove the "Manufacturing of crafts and design goods" category from the animated chart, since its size flattens and hides the transformation of the other sectors. (I do that in the next animated chart, in the next snippet.)

In [ ]:
import plotly.express as px

# Keep World and drop the aggregate total to see the sub-sectors
df_prods_world = products[products['Economy_Label'] == 'World'].copy()
df_subproducts = df_prods_world[df_prods_world['Product_Label'] != 'All creative goods'].copy()
df_subproducts = df_subproducts.sort_values(by=['Year', 'US_at_current_prices_in_millions_Value'])

fig_anim = px.bar(
    df_subproducts,
    x="US_at_current_prices_in_millions_Value",
    y="Product_Label",
    color="Product_Label",
    animation_frame="Year",
    animation_group="Product_Label",
    orientation='h',
    title="Exports by Type of Creative Product (World)",
    labels={'US_at_current_prices_in_millions_Value': 'Millions USD',
           'Product_Label' : 'Creative industry products'},
    range_x=[0, df_subproducts['US_at_current_prices_in_millions_Value'].max() * 1.1]
)

fig_anim.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
fig_anim.update_layout(showlegend=False, height=600, yaxis={'categoryorder': 'total ascending'})
fig_anim.show()

In [ ]:
import plotly.express as px

# --- C. PRODUCTS CHART (ANIMATED) - WITHOUT CRAFTS ---
# 1. Keep World
df_prods_world = products[products['Economy_Label'] == 'World'].copy()

# 2. Filter OUT 'All creative goods' and now also 'Manufacturing of crafts and design goods'
df_subproducts = df_prods_world[
    (~df_prods_world['Product_Label'].isin(['All creative goods', 'Manufacturing of crafts and design goods']))
].copy()

df_subproducts = df_subproducts.sort_values(by=['Year', 'US_at_current_prices_in_millions_Value'])

# 3. Create the chart
fig_anim = px.bar(
    df_subproducts,
    x="US_at_current_prices_in_millions_Value",
    y="Product_Label",
    color="Product_Label",
    animation_frame="Year",
    animation_group="Product_Label",
    orientation='h',
    title="Exports by Type of Creative Product (World) - Excluding Manufacturing",
    labels={
        'US_at_current_prices_in_millions_Value': 'Millions USD',
        'Product_Label' : 'Creative industry products'
    },
    # Adjust the X-axis range dynamically to the new maximum (without crafts)
    range_x=[0, df_subproducts['US_at_current_prices_in_millions_Value'].max() * 1.1]
)

# Speed and layout settings
fig_anim.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
fig_anim.update_layout(showlegend=False, height=600, yaxis={'categoryorder': 'total ascending'})
fig_anim.show()

# 3. Evolution of global exports and imports, crossed with "Cities of Music" designations

Since we have already seen that global music trade is not significantly affected by the "Cities of Music" designations, it is worth exploring whether these international circuits of exchange of identities, urban development, collaboration and social, artistic and cultural understanding also modify this segment of trade at the national level in some way.

This is the finest level of granularity available, since I have not found export/import data at city level, especially considering the large number of designations awarded: 75 cities up to 2023.

In [ ]:
# Group: if a country had 2 designations in a year, join them into a single text
df_events = cities_music.groupby(['Country_Normalized', 'Year'])['City'].agg(
    lambda x: ', '.join(x)
).reset_index()

print(f" Events processed. {len(df_events)} designation milestones ready.")

The `plot_evolution` function works for both exports and imports.

Although the lines are heavily concentrated at the bottom of the chart because of the volume of trade of countries such as the United States or the UK, it works as a first look at the crossing of market trends and "Cities of Music" designations.

By hovering over the chart you can see on the lines which designations were awarded each year, and the color guide helps to locate the country in question.

It is also possible to zoom into the different lines to see the details better, especially where there is a lot of overlap.

By double-clicking a country in the color legend you can isolate that country's line, so you can see how the designations modify (or not) the imports/exports of the country where the city recognized as "City of Music" is located.

In [ ]:
def plot_evolution(df_trade, df_milestones, kind="Exports"):
    # 1. Filter relevant countries
    countries_with_music = df_milestones['Country_Normalized'].unique()
    df_plot = df_trade[df_trade['Country_Normalized'].isin(countries_with_music)].copy()
    
    if df_plot.empty:
        print("No data to plot.")
        return

    df_plot = df_plot.sort_values(['Country_Normalized', 'Year'])
    
    # 2. Use a palette of distinctive colors so they do not repeat and become confusing
    n_countries = len(df_plot['Country_Normalized'].unique())
    my_colors = plotly.colors.sample_colorscale('Turbo', [n/(n_countries -1) if n_countries > 1 else 0 for n in range(n_countries)])

    # 3. Chart
    fig = px.line(
        df_plot,
        x="Year",
        y="US_at_current_prices_in_millions_Value",
        color="Country_Normalized",
        title=f"Evolution of Music {kind} and Designations",
        labels={"US_at_current_prices_in_millions_Value": "Millions USD",
               "Country_Normalized": "Country",
               "Year": "Year"},
        markers=True,
        color_discrete_sequence=my_colors
    )

    # 4. Vertical lines (milestones): dashed line marking the "City of Music" designations per year.
    # Iterate only over the events of the countries present, where Cities of Music have been named.
    df_milestones_filtered = df_milestones[df_milestones['Country_Normalized'].isin(df_plot['Country_Normalized'])]
    
    for _, row in df_milestones_filtered.iterrows():
        fig.add_vline(
            x=row['Year'],
            line_dash="dash",
            line_color="black",
            opacity=0.2
        )

    fig.update_layout(height=600, hovermode="x unified")
    fig.show()

# Run
plot_evolution(mus_exp, df_events, "Exports")
plot_evolution(mus_imp, df_events, "Imports")

# 4. Evolution of exports and imports, broken down by countries with cities named "Cities of Music", in the year of designation.
Next we break the data down by country to observe the trends in their exports and imports per year. I have placed a dashed line in the years in which that country obtained some "City of Music" designation, to gauge whether it was the result of some market variations or whether the designation may have produced those variations.

Sometimes the designation happens when the industry is on the rise, sometimes when it is declining. There are also countries with "Cities of Music" that show zeros in exports and imports, either because the industry is very small or, in most of these cases, because UNCTAD has no data for these countries, mainly African countries, Caribbean islands and Venezuela.

We can thus see that there is no generalizable trend for exports or imports to change categorically after the designations. So it must be stressed that other metrics are needed, capable of measuring the social, cultural and artistic impact of this kind of programme.

In [ ]:
# Music imports by country
df_detail = mus_imp 

country_list = df_detail[df_detail['Country_Normalized'].isin(df_events['Country_Normalized'])]['Country_Normalized'].unique()

for country in country_list:
    # Country data
    df_country = df_detail[df_detail['Country_Normalized'] == country]
    country_events = df_events[df_events['Country_Normalized'] == country]

    fig = px.line(
        df_country,
        x="Year",
        y="US_at_current_prices_in_millions_Value",
        title=f"{country}: Music Imports and Designations",
        markers=True,
        labels={"US_at_current_prices_in_millions_Value": "Imports (Millions USD)"}
    )

    # City markers
    for _, row in country_events.iterrows():
        year = row['Year']
        city_name = row['City']
        
        # Check that the year is within range to avoid breaking the chart
        if df_country['Year'].min() <= year <= df_country['Year'].max():
            fig.add_vline(
                x=year,
                line_dash="dash",
                line_color="green",
                annotation_text=f"City of Music: {city_name}",
                annotation_position="top left",
                annotation_textangle=-90
            )

    fig.update_layout(height=450, showlegend=False)
    fig.show()

In [ ]:
# Music exports by country
df_detail = mus_exp 

country_list = df_detail[df_detail['Country_Normalized'].isin(df_events['Country_Normalized'])]['Country_Normalized'].unique()

for country in country_list:
    # Country data
    df_country = df_detail[df_detail['Country_Normalized'] == country]
    country_events = df_events[df_events['Country_Normalized'] == country]

    fig = px.line(
        df_country,
        x="Year",
        y="US_at_current_prices_in_millions_Value",
        title=f"{country}: Music Exports and Designations",
        markers=True,
        labels={"US_at_current_prices_in_millions_Value": "Exports (Millions USD)"}
    )

    # City markers
    for _, row in country_events.iterrows():
        year = row['Year']
        city_name = row['City']
        
        # Check that the year is within range to avoid breaking the chart
        if df_country['Year'].min() <= year <= df_country['Year'].max():
            fig.add_vline(
                x=year,
                line_dash="dash",
                line_color="green",
                annotation_text=f"City of Music: {city_name}",
                annotation_position="top left",
                annotation_textangle=-90
            )

    fig.update_layout(height=450, showlegend=False)
    fig.show()

# 5. Choropleth map of the "Cities of Music" with music export/import information for the year in which they were named as such by UNESCO
Since we have geolocatable information (we are talking about cities and countries), it is natural to want a visual representation of the distribution of these cities.

To get a better idea of the distribution by country, I have colored those that have "Cities of Music".

In addition, when hovering over the cities marked with a red dot, the year in which they were awarded the title appears, along with the state of their music exports/imports in that year.

In [ ]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd
import plotly.graph_objects as go

# 1. Merge data
df_map = pd.merge(
    cities_music, 
    mus_exp[['Country_Normalized', 'Year', 'US_at_current_prices_in_millions_Value']], 
    on=['Country_Normalized', 'Year'], 
    how='left'
).rename(columns={'US_at_current_prices_in_millions_Value': 'Export_Value'})

df_map = pd.merge(
    df_map, 
    mus_imp[['Country_Normalized', 'Year', 'US_at_current_prices_in_millions_Value']], 
    on=['Country_Normalized', 'Year'], 
    how='left'
).rename(columns={'US_at_current_prices_in_millions_Value': 'Import_Value'})

# 2. Geocoding
geolocator = Nominatim(user_agent="cities_of_music_viz")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

if 'coords_cache' not in globals():
    coords_cache = {}

def get_coords(city, country):
    key = f"{city}, {country}"
    if key in coords_cache:
        return coords_cache[key]
    try:
        loc = geocode(key)
        if loc:
            coords_cache[key] = (loc.latitude, loc.longitude)
            return loc.latitude, loc.longitude
    except:
        return None, None
    return None, None

df_map['coords'] = df_map.apply(lambda x: get_coords(x['City'], x['Country']), axis=1)
df_map[['lat', 'lon']] = pd.DataFrame(df_map['coords'].tolist(), index=df_map.index)
df_map_final = df_map.dropna(subset=['lat', 'lon']).copy()

# 3. PLOTLY PREPARATION: standard English geographic names, not the official UN ones.
# We create a temporary column just for the map: "map_fixes".
map_fixes = {
    'Turkiye': 'Turkey',
    'State of Palestine': 'Palestine',
    'Dem. Rep. of the Congo': 'Democratic Republic of the Congo',
    'Cabo Verde': 'Cape Verde',
    'Viet Nam': 'Vietnam',
    'Republic of Korea': 'South Korea',
    'Russian Federation': 'Russia',
    'Iran (Islamic Republic of)': 'Iran',
    'Bolivia (Plurinational State of)': 'Bolivia',
    'Venezuela (Bolivarian Rep. of)': 'Venezuela'
}

# Apply the correction. If the country is not in the list, it stays the same.
df_map_final['Country_Map_Name'] = df_map_final['Country_Normalized'].replace(map_fixes)

df_map_final['Hover_Text'] = (
    "<b>" + df_map_final['City'] + "</b> (" + df_map_final['Country'] + ")<br>" +
    "Designation: " + df_map_final['Year'].astype(str) + "<br>" +
    "Exp: " + df_map_final['Export_Value'].fillna('No data').astype(str) + " M USD" + "<br>" +
    "Imp: " + df_map_final['Import_Value'].fillna('No data').astype(str) + " M USD"
)

# 4. CREATE THE MAP
fig = go.Figure()

# --- Layer 1: Colored countries (using the corrected names) ---
unique_countries = df_map_final['Country_Map_Name'].unique()

fig.add_trace(go.Choropleth(
    locations=unique_countries,
    locationmode='country names', 
    z=[1] * len(unique_countries),   
    colorscale=[[0, '#a6cee3'], [1, '#a6cee3']],
    showscale=False,
    marker_line_color='white',
    marker_line_width=0.5,
    name='Country with Music Cities',
    hoverinfo='location'
))

# --- Layer 2: City points ---
fig.add_trace(go.Scattergeo(
    lon=df_map_final['lon'],
    lat=df_map_final['lat'],
    text=df_map_final['Hover_Text'],
    mode='markers',
    marker=dict(
        size=8, 
        color='red', 
        line=dict(width=1, color='black'),
        symbol='circle'
    ),
    hoverinfo='text',
    name='Designated City'
))

fig.update_layout(
    title='Map of "Cities of Music" (UNESCO) 2006-2023',
    geo=dict(
        projection_type='natural earth', 
        showland=True,
        landcolor='#f0f0f0',     
        showcountries=True,
        countrycolor='white'
    ),
    margin={"r":0,"t":50,"l":0,"b":0}, 
    height=600
)

fig.show()

# 5a. Corrections to the choropleth map
As can be seen in the previous map, some countries such as Palestine were not properly represented, since Plotly's indexing does not recognize them.

Therefore, to have a fair and adequate visualization, it was necessary to use ISO codes and map using a public GeoJSON available on GitHub: https://raw.githubusercontent.com/johan/world.geo.json/master/countries.geo.json. This way we could have an adequate representation.

In some cases, such as Cabo Verde, the historical series of its music trade is not complete in the UN data. Unfortunately this happens in the year in which Praia was named "City of Music", so it is harder to understand the trade dynamics in that region of the world.
On the other hand, the data for Guinea-Bissau is not covered either by the UN creative industry trade database.

In this regard, these maps let us verify the unequal allocation of titles. It seems almost incredible that the African continent has only six cities (Brazzaville, Kinshasa, Praia, Essaouira, Port Louis and Bissau), while in other regions of the world, such as Europe, almost every country has at least one, and some have up to three, like Italy.

This suggests that the criteria for awarding this category are perhaps biased by city-infrastructure criteria oriented by considerations centered on Western music. It is also hard to explain why countries with millennia-old cultures such as China and Japan have so few (China none, Japan only one, Hamamatsu), since in terms of infrastructure they are also quite competitive.

In [ ]:
import json
from urllib.request import urlopen
import pandas as pd
import plotly.graph_objects as go
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# --- 1. LOAD THE "REAL" MAP (GEOJSON) ---
# We use a public GeoJSON containing the detailed borders of more than 180 countries
url_geojson = "https://raw.githubusercontent.com/johan/world.geo.json/master/countries.geo.json"
with urlopen(url_geojson) as response:
    countries_geojson = json.load(response)

# --- 2. PREPARE DATA (same as before, ensuring ISO codes) ---
# Merge data
df_map = pd.merge(cities_music, mus_exp[['Country_Normalized', 'Year', 'US_at_current_prices_in_millions_Value']], on=['Country_Normalized', 'Year'], how='left').rename(columns={'US_at_current_prices_in_millions_Value': 'Export_Value'})
df_map = pd.merge(df_map, mus_imp[['Country_Normalized', 'Year', 'US_at_current_prices_in_millions_Value']], on=['Country_Normalized', 'Year'], how='left').rename(columns={'US_at_current_prices_in_millions_Value': 'Import_Value'})

# Geocoding
geolocator = Nominatim(user_agent="cities_of_music_viz")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
if 'coords_cache' not in globals(): coords_cache = {}

def get_coords(city, country):
    key = f"{city}, {country}"
    if key in coords_cache: return coords_cache[key]
    try:
        loc = geocode(key)
        if loc:
            coords_cache[key] = (loc.latitude, loc.longitude)
            return loc.latitude, loc.longitude
    except: return None, None
    return None, None

df_map['coords'] = df_map.apply(lambda x: get_coords(x['City'], x['Country']), axis=1)
df_map[['lat', 'lon']] = pd.DataFrame(df_map['coords'].tolist(), index=df_map.index)
df_map_final = df_map.dropna(subset=['lat', 'lon']).copy()

# ISO DICTIONARY
iso_codes = {
    'Turkiye': 'TUR',
    'State of Palestine': 'PSE', 
    'Dem. Rep. of the Congo': 'COD',
    'Cabo Verde': 'CPV',
    'Viet Nam': 'VNM',
    'Republic of Korea': 'KOR',
    'Russian Federation': 'RUS',
    'Iran (Islamic Republic of)': 'IRN',
    'Bolivia (Plurinational State of)': 'BOL',
    'Venezuela (Bolivarian Rep. of)': 'VEN',
    'United Kingdom': 'GBR', 'United States': 'USA', 'Germany': 'DEU', 'France': 'FRA', 
    'Italy': 'ITA', 'Spain': 'ESP', 'Colombia': 'COL', 'Brazil': 'BRA', 'Mexico': 'MEX',
    'Japan': 'JPN', 'China': 'CHN', 'India': 'IND', 'Canada': 'CAN', 'Australia': 'AUS',
    'Argentina': 'ARG', 'Chile': 'CHL', 'Cuba': 'CUB', 'Jamaica': 'JAM', 'Kazakhstan': 'KAZ',
    'Dominican Republic': 'DOM', 'Peru': 'PER', 'Ecuador': 'ECU', 'Sweden': 'SWE', 
    'Norway': 'NOR', 'Finland': 'FIN', 'Denmark': 'DNK', 'Poland': 'POL', 'Hungary': 'HUN',
    'Czechia': 'CZE', 'Austria': 'AUT', 'Switzerland': 'CHE', 'Belgium': 'BEL', 
    'Netherlands': 'NLD', 'Portugal': 'PRT', 'Greece': 'GRC', 'Morocco': 'MAR', 
    'Egypt': 'EGY', 'South Africa': 'ZAF', 'Senegal': 'SEN', 'Kenya': 'KEN', 'Nigeria': 'NGA', 
    'Ghana': 'GHA', 'Azerbaijan': 'AZE', 'Indonesia': 'IDN', 'Thailand': 'THA', 
    'Philippines': 'PHL', 'Malaysia': 'MYS', 'Singapore': 'SGP', 'New Zealand': 'NZL',
    'Bosnia and Herzegovina': 'BIH', 'Serbia': 'SRB', 'Croatia': 'HRV', 'Estonia': 'EST',
    'Latvia': 'LVA', 'Lithuania': 'LTU', 'Ukraine': 'UKR', 'Georgia': 'GEO', 'Congo': 'COG',
    'United Arab Emirates': 'ARE', 'Guinea-Bissau': 'GNB', 'Trinidad and Tobago': 'TTO'
}
df_map_final['ISO_Code'] = df_map_final['Country_Normalized'].map(iso_codes).fillna(df_map_final['Country_Normalized'])

# Hover text
df_map_final['Hover_Text'] = (
    "<b>" + df_map_final['City'] + "</b> (" + df_map_final['Country'] + ")<br>" +
    "Designation: " + df_map_final['Year'].astype(str) + "<br>" +
    "Exp: " + df_map_final['Export_Value'].fillna('No data').astype(str) + " M USD" + "<br>" +
    "Imp: " + df_map_final['Import_Value'].fillna('No data').astype(str) + " M USD"
)

# --- 3. CREATE MAP WITH EXPLICIT GEOJSON ---
fig = go.Figure()

# Layer 1: CHOROPLETH (using the external GeoJSON)
iso_countries = df_map_final['ISO_Code'].unique()

fig.add_trace(go.Choropleth(
    geojson=countries_geojson,           
    locations=iso_countries,                
    featureidkey="id",                   
    z=[1] * len(iso_countries),             
    colorscale=[[0, '#3e95cd'], [1, '#3e95cd']], 
    showscale=False,
    marker_line_color='white',
    marker_line_width=0.5,
    name='Country with Music Cities',
    hoverinfo='location'
))

# Layer 2: RED POINTS
fig.add_trace(go.Scattergeo(
    lon=df_map_final['lon'],
    lat=df_map_final['lat'],
    text=df_map_final['Hover_Text'],
    mode='markers',
    marker=dict(
        size=6, 
        color='#d62728', 
        line=dict(width=1, color='white')
    ),
    hoverinfo='text',
    name='Designated City'
))

# Display settings
fig.update_layout(
    title='Map of Music Cities (UNESCO)',
    geo=dict(
        showframe=False,
        showcoastlines=False,
        projection_type='mercator', 
        visible=False # Hide Plotly's default base map to use only the GeoJSON, with the correct ISO codes
    ),
    margin={"r":0,"t":40,"l":0,"b":0},
    height=650
)

# Mask in gray the countries without "Cities of Music" to highlight those that do
fig.update_geos(
    visible=True, 
    showcountries=True, countrycolor="#e0e0e0", 
    showland=True, landcolor="#f9f9f9"
)

fig.show()

# 6. Comparison of the trade balance (exports - imports): Positioning Analysis (logarithmic scatter plot) and Trade Balance (diverging ranking)
## Points and bars per country, in the year the city was designated "City of Music"
In these charts I set out to observe, with a simple calculation (exports - imports), whether the countries that have "Cities of Music" tend to export or import more music. That is, whether in their trade balance they are positioned more as exporters or importers in the year in which one of their cities obtained the designation.

Already from the scatter plot we see that most of the cities (52 of the 70 with data) have more music goods imports than exports (17 export more than they import, and Norrköping is balanced). Seven cities report zero music exports in their designation year (Almaty, Batoumi, Kinshasa, Kingston, Brazzaville, Santiago de Cuba and Ramallah); a logarithmic axis cannot display zero, so they do not appear in the scatter plot, but they are included in the ranking charts. This of course excludes the large economies such as the United States, the UK, Germany, Italy, etc. But this says more about the size of their economies as a whole than about the importance of music for those societies.
Especially interesting are cities such as Recife, in Brazil, and above all Brno, in the Czech Republic, which stand on the exporter side. The Czech Republic is a country that is not generally recognized internationally for its music, but for other cultural productions.

Since in the scatter plot I had to remove the size of the circles in relation to the amount of exports because it made the chart illegible, I proposed a diverging ranking chart, in which it is possible to see hierarchically, from highest to lowest, the "Cities of Music" with respect to the music goods trade balance of their countries. However, again the extreme values hide the middle values, so the last chart I propose is a symmetric logarithmic adjustment (SYMLOG), which I expand on in the next snippet.

(Five of the 75 cities have no trade data in the sources and are therefore left out of these charts: Praia, Bissau, Caracas, Da Lat and Veliky Novgorod.)

In [ ]:
# 1. Prepare numeric data
df_analysis = df_map_final.copy()
df_analysis['Export_Value'] = pd.to_numeric(df_analysis['Export_Value'], errors='coerce')
df_analysis['Import_Value'] = pd.to_numeric(df_analysis['Import_Value'], errors='coerce')
df_analysis = df_analysis.dropna(subset=['Export_Value', 'Import_Value'])

df_analysis['Balance'] = df_analysis['Export_Value'] - df_analysis['Import_Value']
df_analysis['Profile'] = np.where(df_analysis['Balance'] > 0, 'Mostly Exporter (Surplus)', 'Mostly Importer (Deficit)')

# 2. Logarithmic scatter plot
fig_scatter = px.scatter(
    df_analysis,
    x= "Export_Value",
    y="Import_Value",
    log_x=True, log_y=True,
    color="Profile",
    # size="Export_Value", # size commented out because the outliers hid everything else, but the classification remains
    hover_name="City",
    hover_data=["Country", "Year", "Balance"],
    title="Positioning Map (Logarithmic Scale)",
    color_discrete_map= {'Mostly Exporter (Surplus)': 'purple', 'Mostly Importer (Deficit)': 'green'},
    labels={
        "Export_Value": "Exports (Millions USD)",
        "Import_Value": "Imports (Millions USD)",
        "Profile": "Trade Profile",
        "Balance": "Trade Balance (Millions USD)"
    }
)


# Break-even line
# Only positive values can be placed on a log axis (some countries report 0 exports),
# so the minimum is taken over the positive values to keep the axis range sensible.
positive = pd.concat([df_analysis['Export_Value'], df_analysis['Import_Value']])
positive = positive[positive > 0]
min_val = positive.min()
max_val = positive.max()

fig_scatter.add_shape(
    type="line", x0=min_val, y0=min_val, x1=max_val, y1=max_val,
    line=dict(color="gray", dash="dash")
)
fig_scatter.show()

# 3. Balance ranking
fig_bar = px.bar(
    df_analysis.sort_values('Balance'),
    x="Balance",
    y="City",
    orientation='h',
    color="Balance",
    color_continuous_scale="RdBu",
    title="Trade Balance Ranking in Designation Year",
    height=800,
    hover_data={
        'Country': True,       
        'Year': True,          
        'Balance': ':,.2f',    
        'City': False          
    },
    labels={
        'Year': 'Designation Year',
        'Balance': 'Trade Balance (Millions USD)',
        'City': 'City',
        'Country': 'Country'
    }
)
fig_bar.add_vline(x=0, line_color="black")

fig_bar.update_layout(
    xaxis_title="Deficit <---  (Millions USD)  ---> Surplus",
    yaxis_title="City (Country)"
)

fig_bar.show()

# 6a. Ranking with symmetric logarithmic adjustment
Since the outliers in the "Cities of Music" ranking, in relation to the trade balance of music goods exports/imports of their countries, hid most of the middle values of the diverging bars, I looked into how to improve the visualization so that it made sense. This led me to the "SYMLOG scale", which consists of:
1. Computing the logarithm of the absolute value (to compress the giants).
2. Restoring the original sign (so that the bar sits on the left or right).
3. Readjusting the X axis so that it shows the real numbers ($1 Million, $10 Million, $100 Million) instead of the logarithms (1, 2, 3...).

The chart corroborates what we already said in the previous point: the music market (given above all the "usage rights" of online platforms such as Spotify, Tidal, etc...) concentrates music goods exports unevenly. As a result, most countries are importers of music goods and the "Cities of Music" designation points to values other than economic ones, which does not make them less relevant, but only requires measuring their impact with alternative variables, more focused on the social, the cultural and the qualitative/categorical.

In [ ]:
import numpy as np
import plotly.express as px

# 1. Prepare data 
df_analysis = df_map_final.copy()
df_analysis['Export_Value'] = pd.to_numeric(df_analysis['Export_Value'], errors='coerce')
df_analysis['Import_Value'] = pd.to_numeric(df_analysis['Import_Value'], errors='coerce')
df_analysis = df_analysis.dropna(subset=['Export_Value', 'Import_Value'])
df_analysis['Balance'] = df_analysis['Export_Value'] - df_analysis['Import_Value']

# 2. SYMLOG TRANSFORMATION 
df_analysis['Balance_Symlog'] = np.sign(df_analysis['Balance']) * np.log10(np.abs(df_analysis['Balance']) + 1)
df_analysis = df_analysis.sort_values('Balance_Symlog')

# 3. X-AXIS LABELS
tick_vals = []
tick_text = []
max_exp = int(df_analysis['Balance_Symlog'].abs().max()) + 1

for i in range(1, max_exp + 1):
    val = 10**i
    tick_vals.append(i) 
    tick_text.append(f"${val:,.0f}M")
    tick_vals.append(-i)
    tick_text.append(f"-${val:,.0f}M")

tick_vals.append(0)
tick_text.append("0")

# 4. BAR CHART 
fig_bar = px.bar(
    df_analysis,
    x="Balance_Symlog", # We use the transformed value to draw the bar
    y="City",
    orientation='h',
    color="Balance",    # We use the REAL value for the color
    hover_data={
        'Balance_Symlog': False, # Hide the logarithmic mathematical value
        'Balance': ':,.2f',      # Show the REAL formatted money
        'Country': True,
        'Year': True    
    },
    labels={
        'Year': 'Designation Year',
        'Balance': 'Trade Balance (Millions USD)',
        'City': 'City',
        'Country': 'Country'
    },
    
    color_continuous_scale="RdBu",
    title="Trade Balance Ranking (Symmetric Logarithmic Scale)",
    height=800
)

# Center line
fig_bar.add_vline(x=0, line_color="black", line_width=1)

fig_bar.update_layout(
    xaxis=dict(
        title="Deficit <---  (Logarithmic Scale, USD)  ---> Surplus",
        tickmode='array',
        tickvals=tick_vals,
        ticktext=tick_text
    ),
    yaxis_title="City (Country)"
)

fig_bar.show()

# 7. Conclusions

- Although music goods exports/imports represent traditional music-exporting powers such as the UK well, for countries with less solid economies the indicator collapses in significance.

- This is not entirely bad, since it shows that initiatives such as "Cities of Music" are about something else and are not necessarily subordinate to international markets. Although this does not rule out problems, as we saw in the case of the unequal geographic distribution.

- In addition, the calculation of music goods exports/imports does not really mean that music leaves or enters a country, because the figures also include music/streaming services, concerts, etc.

- Therefore, these metrics speak rather of the dynamism of the markets, but without a logarithmic adjustment they lose proportion. Otherwise, the charts would end up speaking more about the size of each national economy in general than about the music economy specifically.

- The critique of the distribution of "Cities of Music" is very significant. For example, who could say that there are no relevant music cities in Africa? It would be worth reviewing the programme's criteria and trying to refocus them on promoting development and not only on measuring it.

- In addition, it would be ideal to have specific data on the trade movement of music goods at the granular level of cities... The next steps in this kind of research would be to continue exploring other metrics, since by incorporating other "categorical" variables we could get a better idea of the state of music exchanges at the global level.